In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import json

import constants
from model_utils import *
from performance_utils import *

from ast import literal_eval
from pathlib import Path
 
import matplotlib.cm as cm
import ast
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
from pprint import pprint

In [49]:
model_type = constants.RectalCancerStagingData
base_dir = Path.cwd().parent

In [50]:
# Get ORIGINAL DATA
data_path = base_dir / 'data' / constants.RAW_DATA_FILE_NAME
data = pd.read_csv(data_path)
#data = data[['id', 'report_text']]
data.sort_values(by='id', inplace=True, ignore_index=True)


# delete column "posizione" as it is obsolete, renaming "posizione_multiple" in "posizione"
data.drop(columns=['posizione'], inplace=True)
data.rename(columns={'posizione_multiple': 'posizione'}, inplace=True)

sedi_linfonodi = []
for s1, s2 in zip(data.sedi_locoregionali, data.sedi_non_locoregionali):
    sedi_linfonodi.append(str(ast.literal_eval(s1) + ast.literal_eval(s2)))
data['sedi_linfonodi'] = sedi_linfonodi
print(f'Nuova colonna "sedi_linfonodi" creata\n{data.shape = }')

#####################
# Columns aggregation
#####################
# Dettagli infiltrazione organi
infiltrazione_organi_dettagli_new = []
for s in data.infiltrazione_organi_dettagli.fillna(constants.NAN_VALUE):
    dettagli = []
    if s == constants.NAN_VALUE:
        infiltrazione_organi_dettagli_new.append(str(dettagli))
    else:
        if s.strip() == '[object Object]':
            infiltrazione_organi_dettagli_new.append(str(dettagli))
            continue 
        d = ast.literal_eval(s)
        if constants.InfiltrazioneOrganiDettagli.PAVIMENTO_PELVICO.value in d:
            dettagli.append(constants.InfiltrazioneOrganiDettagli.PAVIMENTO_PELVICO.value)
        if ('altro' in d) or ('utero' in d) or ('sacro' in d):
            dettagli.append(constants.InfiltrazioneOrganiDettagli.ALTRO.value)
        infiltrazione_organi_dettagli_new.append(str(dettagli))
data.loc[:, 'infiltrazione_organi_dettagli'] = infiltrazione_organi_dettagli_new


data.set_index('id', inplace=True)
print(f'{data.shape = }')

Nuova colonna "sedi_linfonodi" creata
data.shape = (343, 45)
data.shape = (343, 44)


In [51]:

mistral = load_results_data("new_results_mistral-large-3.jsonl", base_dir/'data'/'inference', model_type)
gpt = load_results_data("new_results_gpt-4.1.jsonl", base_dir/'data'/'inference', model_type)
gpt_tuned = load_results_data("new_results_gpt-4.1_FT.jsonl", base_dir/'data'/'inference', model_type)
opus = load_results_data("new_results_opus-4.6.jsonl", base_dir/'data'/'inference', model_type)
#llama = load_results_data("new_results_llama-3b_FT.jsonl", base_dir/'data'/'inference', model_type)

#assert len(mistral) == len(gpt) == len(opus) == len(gpt_tuned) # == len(llama)


mistral = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in mistral
}
gpt = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in gpt
}
opus = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in opus
}
gpt_tuned = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in gpt_tuned
}

# Controllo
for id in gpt_tuned:
    if mistral[id]['actual'] == gpt[id]['actual'] == opus[id]['actual'] == gpt_tuned[id]['actual']:
        continue
    else:
        print(id)

results = {
    id: {
        'actual':   gpt[id]['actual'],
        'gpt':      gpt[id]['prediction'],
        'opus':     opus[id]['prediction'],
        'mistral':  mistral[id]['prediction'],
        'gpt_tuned': gpt_tuned[id]['prediction'],
    }
    for id in gpt_tuned
}

reports = {id: data.loc[id]['report_text'] for id in gpt_tuned}
profiles = {id: data.loc[id]['profile'] for id in gpt_tuned}

In [52]:
def report_model_to_dataframe(report_model: constants.RectalCancerStagingData, name: str) -> pd.DataFrame:
    multilabel_fields = ['posizione', 'infiltrazione_organi_dettagli', 'sedi_linfonodi']    
    model_dict = report_model.model_dump(mode='json')
    
    for f in multilabel_fields:
        label_list = []
        for l, v in model_dict[f].items():
            if v == constants.Flag.SI.value:
                label_list.append(l)
        model_dict[f] = label_list
    s = pd.Series(model_dict)
    s.name = name
    return s.to_frame()

In [53]:
def original_to_df(id: int, original_data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    result = original_data.loc[id, columns].T
    result.name = 'original'
    return result.to_frame()

In [54]:
def highlight_diff(row, color: str = 'red'):
    ref = row.iloc[1]  # prima colonna come riferimento
    styles = ['', '']  # nessuno stile per la colonna di riferimento
    for val in row.iloc[2:]:  # ultime tre colonne
        if val != ref:
            styles.append(f'background-color: {color}')
        else:
            styles.append('')
    return styles

In [55]:
print(reports.keys())

dict_keys([51, 57, 61, 66, 73, 75, 77, 79, 85, 98, 104, 110, 116, 119, 125, 132, 135, 138, 141, 142, 157, 165, 166, 170, 174, 177, 179, 184, 188, 192, 193, 195, 198, 202, 204, 209, 210, 214, 215, 217, 218, 219, 220, 224, 225, 228, 229, 231, 236, 238, 240, 243, 249, 260, 270, 273, 282, 284, 291, 295, 313, 352, 399, 46, 47, 53, 54, 56, 58, 62, 63, 67, 69, 71, 74, 76, 78, 82, 87, 88, 91, 94, 97, 100, 101, 106, 108, 109, 111, 114, 115, 118, 120, 126, 127, 129, 130, 131, 133, 134, 139, 140, 143, 153, 159, 160, 161, 163, 164, 167, 172, 175, 178, 182, 185, 187, 191, 196, 200, 205, 213, 254, 299, 317, 333, 375, 388, 395])


In [56]:
id = 87
original = original_to_df(id, data, list(constants.RectalCancerStagingData.model_fields.keys()))
dfs = [original] + [report_model_to_dataframe(r, n) for n, r in results[id].items()]
df = pd.concat(dfs, axis=1)
print(data.loc[id].interpretazioni)
styled = df.style.apply(highlight_diff, color='red', axis=1)

nan


In [57]:
print(id, profiles[id])
pprint(reports[id])

87 GuidoImbemba
('SI EVIDENZIA ISPESSIMENTO PARIETALE AGGETTANTE NEL LUME DEL RETTO '
 'MEDIO-ALTO, A CIRCA 7 CM DA OAI, ESTESO PER CIRCA 5 CM, CHE INTERESSA CIRCA '
 'TRE-QUARTI DELLA CIRCONFERENZA DEL VISCERE CON EPICENTRO SUL VERSANTE '
 'ANTERO-LATERALE DI SINISTRA; LA LESIONE INFILTRA A TUTTO SPESSORE LA PARETE '
 'DEL RETTO, CON FINI SEGNI DI ESTENSIONE NEL CELLULARE ADIPOSO MESORETTALE. '
 'TRE LINFONODI, IL MAGGIORE DEI QUALI DEL DIAMETRO DI 6-7 MM, SI OSSERVANO IN '
 'SEDE EMORROIDARIA SUPERIORE, E ALTRI DI MINUTE DIMENSIONI IN SEDE '
 'MESORETTALE IL MAGGIORE DEI QUALI DI 5 MM SUL VERSANTE ANTERIORE SINISTRO '
 '(VEDI IMMAGINI PRINCIPALI). NON FALDE DI VERSAMENTO NELLO SCAVO PELVICO."')


In [58]:
print(id)
display(styled)
print(id, profiles[id])
pprint(reports[id])

87


,original,actual,gpt,opus,mistral,gpt_tuned
morfologia,solido_anulare,solido_anulare,solido_polipoide,solido_polipoide,solido_polipoide,solido_polipoide
ore_inizio,nan,9,8,10,9,12
ore_fine,nan,6,4,6,6,9
spessore_parietale,nan,None,None,None,None,None
estensione_cranio_caudale,50.000000,50,50,50,50,50
distanza_oai,70.000000,70,70,70,70,70
posizione,"['medio', 'alto']","['medio', 'alto']","['medio', 'alto']","['medio', 'alto']","['medio', 'alto']","['medio', 'alto']"
riflessione_peritoneale_anteriore,nan,non_descritto,non_descritto,cavallo,non_descritto,non_descritto
infiltrazione_tessuto_adiposo,si_5mm,si_5mm,si_5mm,si_5mm,si_5mm,si_5mm
infiltrazione_sfinteri,nan,no,no,no,no,no


87 GuidoImbemba
('SI EVIDENZIA ISPESSIMENTO PARIETALE AGGETTANTE NEL LUME DEL RETTO '
 'MEDIO-ALTO, A CIRCA 7 CM DA OAI, ESTESO PER CIRCA 5 CM, CHE INTERESSA CIRCA '
 'TRE-QUARTI DELLA CIRCONFERENZA DEL VISCERE CON EPICENTRO SUL VERSANTE '
 'ANTERO-LATERALE DI SINISTRA; LA LESIONE INFILTRA A TUTTO SPESSORE LA PARETE '
 'DEL RETTO, CON FINI SEGNI DI ESTENSIONE NEL CELLULARE ADIPOSO MESORETTALE. '
 'TRE LINFONODI, IL MAGGIORE DEI QUALI DEL DIAMETRO DI 6-7 MM, SI OSSERVANO IN '
 'SEDE EMORROIDARIA SUPERIORE, E ALTRI DI MINUTE DIMENSIONI IN SEDE '
 'MESORETTALE IL MAGGIORE DEI QUALI DI 5 MM SUL VERSANTE ANTERIORE SINISTRO '
 '(VEDI IMMAGINI PRINCIPALI). NON FALDE DI VERSAMENTO NELLO SCAVO PELVICO."')


In [59]:
print(reports[id])

SI EVIDENZIA ISPESSIMENTO PARIETALE AGGETTANTE NEL LUME DEL RETTO MEDIO-ALTO, A CIRCA 7 CM DA OAI, ESTESO PER CIRCA 5 CM, CHE INTERESSA CIRCA TRE-QUARTI DELLA CIRCONFERENZA DEL VISCERE CON EPICENTRO SUL VERSANTE ANTERO-LATERALE DI SINISTRA; LA LESIONE INFILTRA A TUTTO SPESSORE LA PARETE DEL RETTO, CON FINI SEGNI DI ESTENSIONE NEL CELLULARE ADIPOSO MESORETTALE. TRE LINFONODI, IL MAGGIORE DEI QUALI DEL DIAMETRO DI 6-7 MM, SI OSSERVANO IN SEDE EMORROIDARIA SUPERIORE, E ALTRI DI MINUTE DIMENSIONI IN SEDE MESORETTALE IL MAGGIORE DEI QUALI DI 5 MM SUL VERSANTE ANTERIORE SINISTRO (VEDI IMMAGINI PRINCIPALI). NON FALDE DI VERSAMENTO NELLO SCAVO PELVICO."
